 # AI Ruby on Rails Code Review

 This agent helps to review pull requests in minutes, it suggest best practices, coding convention, security vectors and linters. 

# Install dependencies

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


# Imports

In [3]:
from src.github import download_pr
from src.rag import load_vector_db
from src.reviewer import review
from src.report import markdown_report
from src.parser import parse_diff

# Load DB

In [4]:
db = load_vector_db()

print(f"Knowledge base contains {len(db)} rules.")

Knowledge base contains 8 rules.


# Download Pull Request

In [5]:
PR_URL = "https://github.com/renzodiaz/notes-api/pull/4"

diff = download_pr(PR_URL)

print(f"Downloaded {len(diff)} characters.")
print()
print(diff[:2000])

Downloaded 5625 characters.

diff --git a/Gemfile b/Gemfile
index 00aa1f8..a09c965 100644
--- a/Gemfile
+++ b/Gemfile
@@ -10,7 +10,7 @@ gem "puma", ">= 5.0"
 # gem "jbuilder"
 
 # Use Active Model has_secure_password [https://guides.rubyonrails.org/active_model_basics.html#securepassword]
-# gem "bcrypt", "~> 3.1.7"
+gem "bcrypt", "~> 3.1.7"
 
 # Windows does not include zoneinfo files, so bundle the tzinfo-data gem
 gem "tzinfo-data", platforms: %i[ windows jruby ]
diff --git a/Gemfile.lock b/Gemfile.lock
index bfbf0f0..55df6f9 100644
--- a/Gemfile.lock
+++ b/Gemfile.lock
@@ -77,6 +77,7 @@ GEM
       uri (>= 0.13.1)
     ast (2.4.3)
     base64 (0.3.0)
+    bcrypt (3.1.22)
     bcrypt_pbkdf (1.1.2)
     bigdecimal (4.1.2)
     bootsnap (1.25.0)
@@ -345,6 +346,7 @@ PLATFORMS
   x86_64-linux-musl
 
 DEPENDENCIES
+  bcrypt (~> 3.1.7)
   bootsnap
   brakeman
   bundler-audit
@@ -376,6 +378,7 @@ CHECKSUMS
   activesupport (8.1.3.1) sha256=85458765f25ea48b9019c46b6bb3fa5683197bf4280d9f06710

# Parse the Pull Request

In [6]:
parsed = parse_diff(diff)

print("Ruby files:")
for filename in parsed["ruby_files"]:
    print(f"  - {filename}")

print()
print("Added Ruby code:")
print(parsed["added_lines"][:3000])

Ruby files:
  - app/controllers/api/v1/auth_controller.rb
  - app/controllers/api/v1/secure_controller.rb
  - app/models/user.rb
  - config/routes.rb
  - db/migrate/20260808154603_create_users.rb
  - db/schema.rb
  - test/models/user_test.rb

Added Ruby code:
gem "bcrypt", "~> 3.1.7"
    bcrypt (3.1.22)
  bcrypt (~> 3.1.7)
  bcrypt (3.1.22) sha256=1f0072e88c2d705d94aff7f2c5cb02eb3f1ec4b8368671e19112527489f29032
module Api::V1
    class AuthController < SecureController
        def login
            user = User.find_by(email: params[:email])

            if user && user.authenticate(params[:password])
                render json: { user: user }, status: :ok
            end

            render json: { error: "Invalid email or password" }, status: :unauthorized
        end
    end
end
module Api::V1
    class SecureController < ApplicationController
    end
end
class User < ApplicationRecord
    has_secure_password
end
  namespace :api do
    namespace :v1 do
      post '/login', to: "sec

# Review & Print

In [7]:
review_result = review(diff=diff, db=db)

print(markdown_report(review_result))

# Overall Review

## Summary

This PR adds a basic `User` model with password support and an API login endpoint. The intent is good, but there are a few correctness and security issues in the auth flow, plus some missing database and test safeguards that should be addressed before merging.

## Issues

### [High] Login action always returns unauthorized, even on successful authentication

Category:
Rails / Maintainability

Explanation:
In `Api::V1::AuthController#login`, the success branch renders the user, but execution continues and the unauthorized response is rendered immediately afterward:

```ruby
if user && user.authenticate(params[:password])
  render json: { user: user }, status: :ok
end

render json: { error: "Invalid email or password" }, status: :unauthorized
```

This means the action will always attempt to render twice, which will raise a `DoubleRenderError` in Rails. As written, valid login requests will fail.

Recommendation:
Return early after the success render, or use